# 11 ONNX CPU Optimization + Quantization Candidates v1

10번 노트북의 결론은 명확했다.

```text
Pi 정합성: 통과
Pi latency: 실패
병목: ONNX inference
```

따라서 11번 노트북의 목적은 모델/decoder/steering을 다시 바꾸는 것이 아니다.  
**같은 FP32 ONNX 모델을 CPU에서 더 빠르게 실행할 후보들을 만들고, 그 후보들이 주행 의미를 보존하는지 확인하는 것**이다.

이 노트북의 출력은 12번 Pi 검증으로 보낼 후보 ONNX다.


## 큰 그림

ONNX 파일에는 신경망 forward 계산 그래프만 들어 있다.  
07 decoder와 08 steering은 ONNX 밖의 Python/Numpy 후처리로 유지한다.

```text
FP32 ONNX baseline
  ├─ static QDQ S8S8
  ├─ static QDQ U8S8
  ├─ static QOperator U8S8
  └─ static QOperator U8U8
        ↓
  같은 07 decoder
        ↓
  같은 08 steering
        ↓
  FP32 baseline과 의미 비교 + latency 비교
```

양자화 모델은 raw tensor가 FP32와 완전히 같을 수 없다.  
그래서 이 노트북은 raw diff를 보조 지표로만 보고, 더 중요한 기준은 아래로 둔다.

- lane 개수가 바뀌는가?
- lane 좌표가 얼마나 움직이는가?
- steering mode가 바뀌는가?
- `steer_norm`이 얼마나 달라지는가?
- latency가 실제로 줄어드는가?


## 네 후보의 의미

FP32 ONNX는 후보가 아니라 **비교 기준선**이다.  
모든 후보는 FP32와 같은 입력 이미지를 받고, 같은 07 decoder와 같은 08 steering을 통과한다.

### 1. `static_qdq_s8s8`

CNN용으로 가장 먼저 볼 후보.  
calibration 이미지로 activation 범위를 미리 측정하고, 그래프에 `QuantizeLinear / DeQuantizeLinear` 노드를 넣는다.

```text
activation: signed int8
weight:     signed int8
format:     QDQ
```

장점:

- Conv 연산에서 dynamic보다 의미 있는 속도 개선 가능성이 있다.
- QDQ는 ONNX Runtime에서 흔히 쓰는 표현이다.

단점:

- calibration set이 필요하다.
- FP32와 출력이 달라질 수 있으므로 decode/steering 검증이 필수다.

### 2. `static_qdq_u8s8`

QDQ 방식은 유지하되 activation을 unsigned int8로 둔다.

```text
activation: unsigned int8
weight:     signed int8
format:     QDQ
```

왜 이 후보를 넣는가?

- 이미지 activation은 ReLU 이후 양수 영역이 많아서 U8 activation이 더 잘 맞을 수 있다.
- Pi ARM / ONNX Runtime 조합에서 S8S8보다 U8S8이 더 빠르거나 안정적일 가능성이 있다.
- 실제 결과는 parity + latency로 판단한다.

### 3. `static_qoperator_u8s8`

static quantization이지만 QDQ 노드를 끼워 넣는 대신 `QLinearConv` 같은 quantized operator로 직접 바꾸는 형식이다.

```text
activation: unsigned int8
weight:     signed int8
format:     QOperator
```

장점:

- Pi ARM CPU / ONNX Runtime 빌드에 따라 QDQ보다 빠를 가능성이 있다.

단점:

- 어떤 op가 실제로 최적화되어 있는지는 실행해봐야 안다.
- 정확도 손실이나 runtime 미지원 가능성이 있다.

### 4. `static_qoperator_u8u8`

QOperator 형식은 유지하되 activation과 weight를 둘 다 unsigned int8로 둔다.

```text
activation: unsigned int8
weight:     unsigned int8
format:     QOperator
```

왜 넣는가?

- CPU backend에 따라 U8U8 조합이 U8S8보다 잘 맞을 수 있다.
- 특히 ARM 환경에서는 x86에서 빠른 조합이 그대로 빠르지 않을 수 있다.
- 단, 수치 특성이 달라질 수 있으므로 lane/steering parity를 반드시 봐야 한다.

참고:

- 가장 간단한 `dynamic_int8`도 처음에는 후보로 고려했지만, 이 CLRKDNet ONNX 그래프에서는 ONNX shape inference 단계에서 shape 충돌이 발생했다. 그래서 11번의 주 후보는 실제 생성/실행 가능한 static 계열 중심으로 둔다.
- ONNX Runtime의 optimized graph를 파일로 저장하는 방식도 피한다. NCHWc 같은 CPU 특화 변환이 파일에 박제되면 다른 환경에서 output shape나 호환성 문제가 생길 수 있다. 대신 FP32 baseline과 모든 후보 session에서 runtime graph optimization만 켠다.

따라서 11번은 “이론상 뭐가 좋다”로 결정하지 않고, 네 후보를 실제로 만들어서 같은 데이터로 비교한다.


## Calibration set이란?

static quantization은 activation 값을 int8로 바꾸기 위해 각 layer의 실수값 범위를 미리 알아야 한다.

```text
real_value ≈ scale * (int8_value - zero_point)
```

가중치는 모델 파일 안에 이미 있으므로 범위를 바로 볼 수 있다.  
하지만 activation은 이미지가 모델을 통과할 때 생기는 중간값이라, 실제 이미지를 넣어봐야 범위를 알 수 있다.

그래서 calibration set은 학습 데이터가 아니다.

```text
학습: weight를 업데이트한다.
calibration: weight는 고정하고 activation range만 측정한다.
```

GT 라벨도 필요 없다. 이미지 파일만 있으면 된다.  
이 노트북은 10번 package에 이미 들어 있는 검증 이미지들을 calibration에 사용한다.

```text
val      : 학습/검증 분포
field3   : 실제 주행 시퀀스 분포
holdout  : lane이 없거나 어려운 frame
```

중요한 것은 calibration set이 “정답 라벨이 있는가”가 아니라 **실제 배포 입력 분포를 대표하는가**다.


In [1]:
from pathlib import Path
import json
import shutil
import time
import platform
import traceback

import numpy as np
import pandas as pd
import cv2
import onnx
import onnxruntime as ort

from tqdm.auto import tqdm
from onnxruntime.quantization import (
    CalibrationDataReader,
    CalibrationMethod,
    QuantFormat,
    QuantType,
    quantize_dynamic,
    quantize_static,
)

print("python:", platform.python_version())
print("platform:", platform.platform())
print("onnx:", onnx.__version__)
print("onnxruntime:", ort.__version__)


python: 3.10.18
platform: Windows-10-10.0.26200-SP0
onnx: 1.21.0
onnxruntime: 1.23.2


In [2]:
# ----- Path configuration -----
BASE = Path(r"~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild")

SOURCE_ONNX = BASE / "review_outputs" / "09_onnx_export_parity_v1" / "models" / "MapLane_LocalFit_Field12_v1_best_opset17_static_b1.onnx"
SOURCE_ONNX_DATA = SOURCE_ONNX.with_name(SOURCE_ONNX.name + ".data")

PKG10 = BASE / "review_outputs" / "10_pi_runtime_latency_sequence_validation_v1" / "pkg"
NOTEBOOK10 = BASE / "notebooks" / "10_pi_runtime_latency_sequence_validation_v1.ipynb"

OUT_DIR = BASE / "review_outputs" / "11_onnx_cpu_quantization_candidates_v1"
MODELS_DIR = OUT_DIR / "models"
TABLES_DIR = OUT_DIR / "tables"
REPORTS_DIR = OUT_DIR / "reports"
VIS_DIR = OUT_DIR / "visuals"

for p in [OUT_DIR, MODELS_DIR, TABLES_DIR, REPORTS_DIR, VIS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

assert SOURCE_ONNX.exists(), SOURCE_ONNX
assert SOURCE_ONNX_DATA.exists(), SOURCE_ONNX_DATA
assert PKG10.exists(), PKG10
assert NOTEBOOK10.exists(), NOTEBOOK10

print("OUT_DIR:", OUT_DIR)
print("SOURCE_ONNX:", SOURCE_ONNX)
print("PKG10:", PKG10)


OUT_DIR: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\11_onnx_cpu_quantization_candidates_v1
SOURCE_ONNX: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\09_onnx_export_parity_v1\models\MapLane_LocalFit_Field12_v1_best_opset17_static_b1.onnx
PKG10: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\10_pi_runtime_latency_sequence_validation_v1\pkg


In [3]:
# ----- Experiment knobs -----
# None이면 10번 package 안의 해당 record를 전부 사용한다.
CALIB_RECORD_LIMIT = None
PARITY_RECORD_LIMIT = None
SEQUENCE_RECORD_LIMIT = 160
LATENCY_RECORD_LIMIT = 160

ORT_THREADS_LOCAL = None  # None = ORT default. Pi에서는 10번에서 threads=4를 썼다.
ORT_WARMUP_RUNS = 5

OP_TYPES_TO_QUANTIZE = ["Conv", "MatMul", "Gemm"]

# 양자화 후보가 조금 다르게 나오는 것은 정상이다.
# 여기서는 "로컬에서 12번 Pi 검증으로 보낼 수 있을 정도인가"를 보는 넉넉한 기준이다.
MEAN_LANE_DIST_SOFT_LIMIT_PX = 2.0
MAX_STEER_DIFF_SOFT_LIMIT = 0.05
MODE_MISMATCH_SOFT_LIMIT = 0
COUNT_MISMATCH_SOFT_LIMIT = 0

print("CALIB_RECORD_LIMIT:", CALIB_RECORD_LIMIT)
print("PARITY_RECORD_LIMIT:", PARITY_RECORD_LIMIT)
print("SEQUENCE_RECORD_LIMIT:", SEQUENCE_RECORD_LIMIT)
print("LATENCY_RECORD_LIMIT:", LATENCY_RECORD_LIMIT)


CALIB_RECORD_LIMIT: None
PARITY_RECORD_LIMIT: None
SEQUENCE_RECORD_LIMIT: 160
LATENCY_RECORD_LIMIT: 160


## 10번 Runtime Core 재사용

11번에서 decoder/steering 함수를 새로 베껴 쓰면, 10번과 미세하게 달라질 위험이 있다.  
그래서 이 노트북은 10번 노트북의 Runtime Core 셀을 그대로 실행해서 아래 함수를 재사용한다.

- `preprocess_bgr_for_model`
- `decode_raw_to_lanes`
- `init_drive_memory`
- `update_drive`
- `imread_bgr`

즉, 11번에서 비교하는 것은 **모델 후보만**이고, decoder/steering 구현은 10번과 동일하게 유지한다.


In [4]:
# ----- Minimal globals required by the 10 notebook runtime core -----
IS_PI = False
RAW_W, RAW_H = 1296, 972
CUT_HEIGHT = 445
IMG_W, IMG_H = 800, 320
NUM_PRIORS = 192
OUTPUT_DIM = 78
N_OFFSETS = 72
N_STRIPS = N_OFFSETS - 1
SAMPLE_Y = list(range(971, 444, -20))
IMAGE_CENTER_X = RAW_W / 2.0
DEFAULT_PAIR_BONUS_PX = 60.0
DEFAULT_PI_ORT_THREADS = 4
REQUIRE_SCIPY_FOR_DECODER = True
ORT_WARMUP_RUNS = int(ORT_WARMUP_RUNS)

def fs_path(path):
    path = Path(path)
    s = str(path)
    if platform.system().lower().startswith("win"):
        try:
            s = str(path.resolve())
        except Exception:
            s = str(path)
        if not s.startswith("\\\\?\\"):
            s = "\\\\?\\" + s
    return s

def exists_fs(path):
    try:
        return Path(path).exists()
    except OSError:
        return Path(fs_path(path)).exists()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, obj):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")

def imread_bgr(path):
    data = np.fromfile(fs_path(path), dtype=np.uint8)
    img = cv2.imdecode(data, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    return img

def imwrite_bgr(path, img):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    ok, buf = cv2.imencode(Path(path).suffix or ".jpg", img)
    if not ok:
        raise RuntimeError(f"cv2.imencode failed: {path}")
    buf.tofile(fs_path(path))

try:
    from scipy.interpolate import InterpolatedUnivariateSpline
    HAS_SCIPY = True
except Exception as exc:
    HAS_SCIPY = False
    if REQUIRE_SCIPY_FOR_DECODER:
        raise RuntimeError("SciPy is required for official-compatible Lane.to_array spline resampling.") from exc

nb10 = json.loads(NOTEBOOK10.read_text(encoding="utf-8"))
runtime_core = None
for cell in nb10["cells"]:
    if cell.get("cell_type") == "code":
        src = "".join(cell.get("source", []))
        if src.lstrip().startswith("# ----- Contracts and ONNX Runtime -----"):
            runtime_core = src
            break
assert runtime_core is not None, "10 notebook runtime core cell was not found."
exec(compile(runtime_core, "10_runtime_core", "exec"), globals())
print("Loaded runtime core from:", NOTEBOOK10)


Loaded runtime core from: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\notebooks\10_pi_runtime_latency_sequence_validation_v1.ipynb


In [5]:
# ----- Load records and contracts from the 10 package -----
decode_contract, driving_contract, _, _ = load_package_contracts(PKG10)
records = pd.read_csv(PKG10 / "t" / "records_manifest.csv")
records["image_path"] = records["image_rel"].apply(lambda rel: str(PKG10 / rel))

parity_records = records[records["role"] == "parity"].sort_values(["set", "order"]).copy()
sequence_records = records[records["role"] == "sequence"].sort_values(["order"]).copy()

if PARITY_RECORD_LIMIT is not None:
    parity_records = parity_records.head(int(PARITY_RECORD_LIMIT)).copy()
if SEQUENCE_RECORD_LIMIT is not None:
    sequence_records = sequence_records.head(int(SEQUENCE_RECORD_LIMIT)).copy()

calib_records = records.drop_duplicates("image_rel").sort_values(["set", "role", "order"]).copy()
if CALIB_RECORD_LIMIT is not None:
    calib_records = calib_records.head(int(CALIB_RECORD_LIMIT)).copy()

latency_records = sequence_records.copy()
if LATENCY_RECORD_LIMIT is not None:
    latency_records = latency_records.head(int(LATENCY_RECORD_LIMIT)).copy()

print("records total:", len(records))
print("calibration records:", len(calib_records))
print("parity records:", len(parity_records))
print("sequence records:", len(sequence_records))
print("latency records:", len(latency_records))
print(records.groupby(["set", "role"]).size())


records total: 356
calibration records: 356
parity records: 116
sequence records: 160
latency records: 160
set      role    
field3   parity       80
         sequence    240
holdout  parity       24
val      parity       12
dtype: int64


In [6]:
class ImageCalibrationDataReader(CalibrationDataReader):
    # CalibrationDataReader for ONNX Runtime static quantization.
    # 이 객체는 label/GT를 읽지 않는다.
    # image -> preprocess -> {input_name: tensor}만 제공한다.
    def __init__(self, image_paths, input_name):
        self.image_paths = list(image_paths)
        self.input_name = input_name
        self._iter = None

    def get_next(self):
        if self._iter is None:
            self._iter = iter(self.image_paths)
        try:
            p = next(self._iter)
        except StopIteration:
            return None
        bgr = imread_bgr(p)
        return {self.input_name: preprocess_bgr_for_model(bgr)}

    def rewind(self):
        self._iter = None

def make_session_from_model(model_path, intra_op_num_threads=ORT_THREADS_LOCAL):
    so = ort.SessionOptions()
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    if intra_op_num_threads is not None:
        so.intra_op_num_threads = int(intra_op_num_threads)
    session = ort.InferenceSession(fs_path(model_path), sess_options=so, providers=["CPUExecutionProvider"])
    return session, session.get_inputs()[0].name, session.get_outputs()[0].name

def warmup_session(session, input_name, image_paths, runs=ORT_WARMUP_RUNS):
    if not image_paths:
        return
    bgr = imread_bgr(image_paths[0])
    inp = preprocess_bgr_for_model(bgr)
    for _ in range(int(runs)):
        session.run(None, {input_name: inp})

def run_raw_from_session(session, input_name, output_name, bgr):
    inp = preprocess_bgr_for_model(bgr)
    out = session.run([output_name], {input_name: inp})[0]
    assert out.shape == (1, NUM_PRIORS, OUTPUT_DIM), out.shape
    return out[0].astype(np.float32)

def file_mb(path):
    return Path(path).stat().st_size / (1024 * 1024)


## 후보 모델 생성

이 셀은 네 후보를 만든다.

실패할 수 있는 후보는 실패로 기록하고 넘어간다.  
예를 들어 QOperator 후보가 현재 ONNX Runtime에서 어떤 op를 지원하지 않으면 생성 또는 실행 단계에서 실패할 수 있다.

중요한 것은 실패 자체가 나쁜 게 아니라, **Pi/ORT 조합에서 실제로 쓸 수 있는 후보만 남기는 것**이다.


In [7]:
def prepare_source_model():
    local_onnx = MODELS_DIR / SOURCE_ONNX.name
    local_data = MODELS_DIR / SOURCE_ONNX_DATA.name
    shutil.copy2(SOURCE_ONNX, local_onnx)
    shutil.copy2(SOURCE_ONNX_DATA, local_data)
    return local_onnx

def generate_quantization_candidates():
    source_model = prepare_source_model()
    probe_session, input_name, _ = make_session_from_model(source_model)
    image_paths = calib_records["image_path"].tolist()

    candidate_specs = [
        {
            "name": "static_qdq_s8s8",
            "kind": "static_quantization",
            "path": MODELS_DIR / "static_qdq_s8s8.onnx",
            "description": "Static QDQ quantization, activation QInt8 + weight QInt8.",
        },
        {
            "name": "static_qdq_u8s8",
            "kind": "static_quantization",
            "path": MODELS_DIR / "static_qdq_u8s8.onnx",
            "description": "Static QDQ quantization, activation QUInt8 + weight QInt8.",
        },
        {
            "name": "static_qoperator_u8s8",
            "kind": "static_quantization",
            "path": MODELS_DIR / "static_qoperator_u8s8.onnx",
            "description": "Static QOperator quantization, activation QUInt8 + weight QInt8.",
        },
        {
            "name": "static_qoperator_u8u8",
            "kind": "static_quantization",
            "path": MODELS_DIR / "static_qoperator_u8u8.onnx",
            "description": "Static QOperator quantization, activation QUInt8 + weight QUInt8.",
        },
    ]

    rows = []
    for spec in candidate_specs:
        name = spec["name"]
        out = spec["path"]
        t0 = time.perf_counter()
        status = "ok"
        error = ""
        try:
            if out.exists():
                out.unlink()
            if name == "static_qdq_s8s8":
                reader = ImageCalibrationDataReader(image_paths, input_name)
                quantize_static(
                    model_input=source_model,
                    model_output=out,
                    calibration_data_reader=reader,
                    quant_format=QuantFormat.QDQ,
                    op_types_to_quantize=OP_TYPES_TO_QUANTIZE,
                    per_channel=True,
                    activation_type=QuantType.QInt8,
                    weight_type=QuantType.QInt8,
                    calibrate_method=CalibrationMethod.MinMax,
                    use_external_data_format=False,
                    extra_options={"ActivationSymmetric": True, "WeightSymmetric": True},
                )
            elif name == "static_qdq_u8s8":
                reader = ImageCalibrationDataReader(image_paths, input_name)
                quantize_static(
                    model_input=source_model,
                    model_output=out,
                    calibration_data_reader=reader,
                    quant_format=QuantFormat.QDQ,
                    op_types_to_quantize=OP_TYPES_TO_QUANTIZE,
                    per_channel=False,
                    activation_type=QuantType.QUInt8,
                    weight_type=QuantType.QInt8,
                    calibrate_method=CalibrationMethod.MinMax,
                    use_external_data_format=False,
                )
            elif name == "static_qoperator_u8s8":
                reader = ImageCalibrationDataReader(image_paths, input_name)
                quantize_static(
                    model_input=source_model,
                    model_output=out,
                    calibration_data_reader=reader,
                    quant_format=QuantFormat.QOperator,
                    op_types_to_quantize=OP_TYPES_TO_QUANTIZE,
                    per_channel=False,
                    activation_type=QuantType.QUInt8,
                    weight_type=QuantType.QInt8,
                    calibrate_method=CalibrationMethod.MinMax,
                    use_external_data_format=False,
                )
            elif name == "static_qoperator_u8u8":
                reader = ImageCalibrationDataReader(image_paths, input_name)
                quantize_static(
                    model_input=source_model,
                    model_output=out,
                    calibration_data_reader=reader,
                    quant_format=QuantFormat.QOperator,
                    op_types_to_quantize=OP_TYPES_TO_QUANTIZE,
                    per_channel=False,
                    activation_type=QuantType.QUInt8,
                    weight_type=QuantType.QUInt8,
                    calibrate_method=CalibrationMethod.MinMax,
                    use_external_data_format=False,
                )
            else:
                raise ValueError(name)
            assert out.exists(), out
        except Exception as exc:
            status = "error"
            error = "".join(traceback.format_exception_only(type(exc), exc)).strip()

        rows.append({
            "name": name,
            "kind": spec["kind"],
            "path": str(out),
            "description": spec["description"],
            "status": status,
            "error": error,
            "create_sec": time.perf_counter() - t0,
            "size_mb": file_mb(out) if out.exists() else np.nan,
        })
        print(name, status, error if error else f"{rows[-1]['size_mb']:.2f} MB")

    df = pd.DataFrame(rows)
    df.to_csv(TABLES_DIR / "candidate_generation.csv", index=False, encoding="utf-8-sig")
    return df

candidate_generation = generate_quantization_candidates()
candidate_generation


static_qdq_s8s8 ok 11.60 MB


static_qdq_u8s8 ok 11.52 MB


static_qoperator_u8s8 ok 11.40 MB


static_qoperator_u8u8 ok 11.40 MB


,name,kind,path,description,status,error,create_sec,size_mb
0,static_qdq_s8s8,static_quantization,~\02_Projects\University\26-1_Em...,"Static QDQ quantization, activation QInt8 + we...",ok,,51.638982,11.599054
1,static_qdq_u8s8,static_quantization,~\02_Projects\University\26-1_Em...,"Static QDQ quantization, activation QUInt8 + w...",ok,,54.141419,11.524275
2,static_qoperator_u8s8,static_quantization,~\02_Projects\University\26-1_Em...,"Static QOperator quantization, activation QUIn...",ok,,57.784991,11.395810
3,static_qoperator_u8u8,static_quantization,~\02_Projects\University\26-1_Em...,"Static QOperator quantization, activation QUIn...",ok,,58.545397,11.395830


## 후보 평가 방식

평가는 FP32 ONNX를 기준으로 한다.  
각 후보 모델에 같은 이미지를 넣고 아래를 비교한다.

### 1. Raw parity

신경망 출력 tensor 자체의 차이다.  
양자화 모델은 raw가 달라지는 것이 정상이라, 이 값은 참고 지표다.

### 2. Decode parity

07 decoder를 적용한 lane 결과가 얼마나 달라졌는지 본다.

- lane count mismatch
- 같은 순서 lane끼리 평균 거리
- 최대 lane 거리

### 3. Steering parity

08 steering을 field3 sequence에 적용했을 때 주행 명령이 얼마나 달라졌는지 본다.

- mode mismatch
- `steer_norm` 차이
- center/heading 차이

### 4. Latency

로컬 CPU에서 먼저 속도를 비교한다.  
단, 최종 속도 판단은 12번 Pi 검증에서 한다.


In [8]:
def lane_pair_distance(lane_a, lane_b):
    pa = np.asarray(lane_a["points"], dtype=np.float32)
    pb = np.asarray(lane_b["points"], dtype=np.float32)
    if len(pa) == 0 or len(pb) == 0:
        return np.nan
    # 같은 sample_y를 쓰지만 valid point 수가 다를 수 있으므로 y 기준으로 가까운 점만 비교한다.
    dists = []
    for x, y in pa:
        j = int(np.argmin(np.abs(pb[:, 1] - y)))
        if abs(float(pb[j, 1] - y)) <= 1.0:
            dists.append(abs(float(pb[j, 0] - x)))
    return float(np.mean(dists)) if dists else np.nan

def compare_lane_sets(base_lanes, cand_lanes):
    count_mismatch = int(len(base_lanes) != len(cand_lanes))
    pair_dists = []
    for a, b in zip(base_lanes, cand_lanes):
        d = lane_pair_distance(a, b)
        if np.isfinite(d):
            pair_dists.append(d)
    return {
        "base_count": len(base_lanes),
        "cand_count": len(cand_lanes),
        "count_mismatch": count_mismatch,
        "mean_pair_dist_px": float(np.mean(pair_dists)) if pair_dists else np.nan,
        "max_pair_dist_px": float(np.max(pair_dists)) if pair_dists else np.nan,
    }

def evaluate_candidate_model(name, model_path, fp32_session_bundle):
    fp32_session, fp32_input, fp32_output = fp32_session_bundle
    out_rows_raw = []
    out_rows_decode = []
    out_rows_latency = []
    out_rows_steer = []

    try:
        session, input_name, output_name = make_session_from_model(model_path)
        image_paths = latency_records["image_path"].tolist()
        warmup_session(session, input_name, image_paths)
    except Exception as exc:
        return {
            "name": name,
            "status": "runtime_error",
            "error": "".join(traceback.format_exception_only(type(exc), exc)).strip(),
            "raw_rows": pd.DataFrame(),
            "decode_rows": pd.DataFrame(),
            "steer_rows": pd.DataFrame(),
            "latency_rows": pd.DataFrame(),
        }

    fp32_lanes_by_key = {}
    cand_lanes_by_key = {}

    for _, rec in tqdm(parity_records.iterrows(), total=len(parity_records), desc=f"{name} parity"):
        key = rec["key"]
        bgr = imread_bgr(rec["image_path"])
        fp32_raw = run_raw_from_session(fp32_session, fp32_input, fp32_output, bgr)
        cand_raw = run_raw_from_session(session, input_name, output_name, bgr)
        diff = np.abs(fp32_raw - cand_raw)
        fp32_lanes = decode_raw_to_lanes(fp32_raw, decode_contract)
        cand_lanes = decode_raw_to_lanes(cand_raw, decode_contract)
        fp32_lanes_by_key[key] = fp32_lanes
        cand_lanes_by_key[key] = cand_lanes
        lane_cmp = compare_lane_sets(fp32_lanes, cand_lanes)

        out_rows_raw.append({
            "candidate": name,
            "key": key,
            "set": rec["set"],
            "max_abs_diff": float(diff.max()),
            "mean_abs_diff": float(diff.mean()),
            "p99_abs_diff": float(np.quantile(diff, 0.99)),
        })
        out_rows_decode.append({
            "candidate": name,
            "key": key,
            "set": rec["set"],
            **lane_cmp,
        })

    base_mem = init_drive_memory()
    cand_mem = init_drive_memory()
    for _, rec in tqdm(sequence_records.iterrows(), total=len(sequence_records), desc=f"{name} steering"):
        key = rec["key"]
        bgr = imread_bgr(rec["image_path"])
        fp32_raw = run_raw_from_session(fp32_session, fp32_input, fp32_output, bgr)
        cand_raw = run_raw_from_session(session, input_name, output_name, bgr)
        fp32_lanes = decode_raw_to_lanes(fp32_raw, decode_contract)
        cand_lanes = decode_raw_to_lanes(cand_raw, decode_contract)
        base_row = update_drive(fp32_lanes, base_mem, driving_contract)
        cand_row = update_drive(cand_lanes, cand_mem, driving_contract)
        out_rows_steer.append({
            "candidate": name,
            "key": key,
            "order": int(rec["order"]),
            "base_mode": base_row["effective_mode"],
            "cand_mode": cand_row["effective_mode"],
            "mode_mismatch": int(base_row["effective_mode"] != cand_row["effective_mode"]),
            "base_steer": float(base_row["steer_norm"]),
            "cand_steer": float(cand_row["steer_norm"]),
            "steer_abs_diff": abs(float(base_row["steer_norm"]) - float(cand_row["steer_norm"])),
            "center_abs_diff": abs(float(base_row["smoothed_center_x"]) - float(cand_row["smoothed_center_x"])),
            "heading_abs_diff": abs(float(base_row["smoothed_heading"]) - float(cand_row["smoothed_heading"])),
        })

    for _, rec in tqdm(latency_records.iterrows(), total=len(latency_records), desc=f"{name} latency"):
        bgr = imread_bgr(rec["image_path"])
        t0 = time.perf_counter()
        inp = preprocess_bgr_for_model(bgr)
        t1 = time.perf_counter()
        raw = session.run([output_name], {input_name: inp})[0][0]
        t2 = time.perf_counter()
        lanes = decode_raw_to_lanes(raw.astype(np.float32), decode_contract)
        t3 = time.perf_counter()
        out_rows_latency.append({
            "candidate": name,
            "key": rec["key"],
            "preprocess_ms": (t1 - t0) * 1000,
            "inference_ms": (t2 - t1) * 1000,
            "decode_ms": (t3 - t2) * 1000,
            "pipeline_ms": (t3 - t0) * 1000,
            "lane_count": len(lanes),
        })

    return {
        "name": name,
        "status": "ok",
        "error": "",
        "raw_rows": pd.DataFrame(out_rows_raw),
        "decode_rows": pd.DataFrame(out_rows_decode),
        "steer_rows": pd.DataFrame(out_rows_steer),
        "latency_rows": pd.DataFrame(out_rows_latency),
    }


In [9]:
# ----- Evaluate all generated candidates -----
source_model = MODELS_DIR / SOURCE_ONNX.name
fp32_session_bundle = make_session_from_model(source_model)
warmup_session(fp32_session_bundle[0], fp32_session_bundle[1], latency_records["image_path"].tolist())

eval_results = []
raw_all, decode_all, steer_all, latency_all = [], [], [], []

ok_candidates = candidate_generation[candidate_generation["status"] == "ok"].copy()
for _, row in ok_candidates.iterrows():
    result = evaluate_candidate_model(row["name"], Path(row["path"]), fp32_session_bundle)
    eval_results.append({"name": result["name"], "status": result["status"], "error": result["error"]})
    if not result["raw_rows"].empty:
        raw_all.append(result["raw_rows"])
    if not result["decode_rows"].empty:
        decode_all.append(result["decode_rows"])
    if not result["steer_rows"].empty:
        steer_all.append(result["steer_rows"])
    if not result["latency_rows"].empty:
        latency_all.append(result["latency_rows"])

raw_df = pd.concat(raw_all, ignore_index=True) if raw_all else pd.DataFrame()
decode_df = pd.concat(decode_all, ignore_index=True) if decode_all else pd.DataFrame()
steer_df = pd.concat(steer_all, ignore_index=True) if steer_all else pd.DataFrame()
latency_df = pd.concat(latency_all, ignore_index=True) if latency_all else pd.DataFrame()

raw_df.to_csv(TABLES_DIR / "candidate_raw_parity.csv", index=False, encoding="utf-8-sig")
decode_df.to_csv(TABLES_DIR / "candidate_decode_parity.csv", index=False, encoding="utf-8-sig")
steer_df.to_csv(TABLES_DIR / "candidate_steering_parity.csv", index=False, encoding="utf-8-sig")
latency_df.to_csv(TABLES_DIR / "candidate_latency.csv", index=False, encoding="utf-8-sig")

pd.DataFrame(eval_results).to_csv(TABLES_DIR / "candidate_eval_status.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(eval_results)


static_qdq_s8s8 parity:   0%|          | 0/116 [00:00<?, ?it/s]

static_qdq_s8s8 steering:   0%|          | 0/160 [00:00<?, ?it/s]

static_qdq_s8s8 latency:   0%|          | 0/160 [00:00<?, ?it/s]

static_qdq_u8s8 parity:   0%|          | 0/116 [00:00<?, ?it/s]

static_qdq_u8s8 steering:   0%|          | 0/160 [00:00<?, ?it/s]

...

static_qoperator_u8s8 latency:   0%|          | 0/160 [00:00<?, ?it/s]

static_qoperator_u8u8 parity:   0%|          | 0/116 [00:00<?, ?it/s]

static_qoperator_u8u8 steering:   0%|          | 0/160 [00:00<?, ?it/s]

static_qoperator_u8u8 latency:   0%|          | 0/160 [00:00<?, ?it/s]

,name,status,error
0,static_qdq_s8s8,ok,
1,static_qdq_u8s8,ok,
2,static_qoperator_u8s8,ok,
3,static_qoperator_u8u8,ok,


In [10]:
def q95(s):
    return float(s.quantile(0.95)) if len(s) else np.nan

summary_rows = []
for _, gen in candidate_generation.iterrows():
    name = gen["name"]
    status_rows = [r for r in eval_results if r["name"] == name]
    eval_status = status_rows[0]["status"] if status_rows else ("not_run" if gen["status"] == "ok" else "not_generated")
    eval_error = status_rows[0]["error"] if status_rows else gen.get("error", "")
    raw_sub = raw_df[raw_df["candidate"] == name] if not raw_df.empty else pd.DataFrame()
    dec_sub = decode_df[decode_df["candidate"] == name] if not decode_df.empty else pd.DataFrame()
    ste_sub = steer_df[steer_df["candidate"] == name] if not steer_df.empty else pd.DataFrame()
    lat_sub = latency_df[latency_df["candidate"] == name] if not latency_df.empty else pd.DataFrame()

    decode_count_mismatch = int(dec_sub["count_mismatch"].sum()) if not dec_sub.empty else np.nan
    steer_mode_mismatch = int(ste_sub["mode_mismatch"].sum()) if not ste_sub.empty else np.nan
    mean_lane_dist = float(dec_sub["mean_pair_dist_px"].dropna().mean()) if not dec_sub.empty else np.nan
    max_lane_dist = float(dec_sub["max_pair_dist_px"].dropna().max()) if not dec_sub.empty else np.nan
    max_steer_diff = float(ste_sub["steer_abs_diff"].max()) if not ste_sub.empty else np.nan
    mean_pipeline = float(lat_sub["pipeline_ms"].mean()) if not lat_sub.empty else np.nan
    fps_mean = float(1000.0 / mean_pipeline) if np.isfinite(mean_pipeline) and mean_pipeline > 0 else np.nan

    meaning_pass = (
        eval_status == "ok"
        and decode_count_mismatch <= COUNT_MISMATCH_SOFT_LIMIT
        and steer_mode_mismatch <= MODE_MISMATCH_SOFT_LIMIT
        and (not np.isfinite(mean_lane_dist) or mean_lane_dist <= MEAN_LANE_DIST_SOFT_LIMIT_PX)
        and (not np.isfinite(max_steer_diff) or max_steer_diff <= MAX_STEER_DIFF_SOFT_LIMIT)
    )

    summary_rows.append({
        "candidate": name,
        "generation_status": gen["status"],
        "eval_status": eval_status,
        "meaning_pass": bool(meaning_pass),
        "size_mb": gen["size_mb"],
        "raw_max_abs_diff": float(raw_sub["max_abs_diff"].max()) if not raw_sub.empty else np.nan,
        "raw_mean_abs_diff_max": float(raw_sub["mean_abs_diff"].max()) if not raw_sub.empty else np.nan,
        "decode_count_mismatch": decode_count_mismatch,
        "decode_mean_pair_dist_px": mean_lane_dist,
        "decode_max_pair_dist_px": max_lane_dist,
        "steering_mode_mismatch": steer_mode_mismatch,
        "steering_max_abs_diff": max_steer_diff,
        "latency_pipeline_mean_ms": mean_pipeline,
        "latency_pipeline_p95_ms": q95(lat_sub["pipeline_ms"]) if not lat_sub.empty else np.nan,
        "latency_inference_mean_ms": float(lat_sub["inference_ms"].mean()) if not lat_sub.empty else np.nan,
        "fps_mean": fps_mean,
        "error": eval_error if eval_error else gen.get("error", ""),
        "path": gen["path"],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(TABLES_DIR / "candidate_eval_summary.csv", index=False, encoding="utf-8-sig")
summary_df.sort_values(["meaning_pass", "latency_pipeline_mean_ms"], ascending=[False, True])


,candidate,generation_status,eval_status,meaning_pass,size_mb,raw_max_abs_diff,raw_mean_abs_diff_max,decode_count_mismatch,decode_mean_pair_dist_px,decode_max_pair_dist_px,steering_mode_mismatch,steering_max_abs_diff,latency_pipeline_mean_ms,latency_pipeline_p95_ms,latency_inference_mean_ms,fps_mean,error,path
1,static_qdq_u8s8,ok,ok,False,11.524275,1370.558716,2.798258,8,59.485999,1277.873657,6,0.142652,45.482649,75.058365,40.471917,21.986406,,~\02_Projects\University\26-1_Em...
2,static_qoperator_u8s8,ok,ok,False,11.395810,1370.787354,2.798177,10,47.384908,1250.321606,14,0.167616,49.598492,95.864185,45.468721,20.161903,,~\02_Projects\University\26-1_Em...
3,static_qoperator_u8u8,ok,ok,False,11.395830,1371.010498,2.799003,16,55.053058,865.068497,11,0.166573,70.052956,113.030810,65.518829,14.274915,,~\02_Projects\University\26-1_Em...
0,static_qdq_s8s8,ok,ok,False,11.599054,1416.275513,2.889508,14,55.644877,952.166378,10,0.162032,79.746152,124.727725,75.447949,12.539790,,~\02_Projects\University\26-1_Em...


## 후보 선택 규칙

11번의 선택은 최종 배포 확정이 아니다.  
여기서 고른 후보는 **12번 Pi runtime 검증으로 보낼 후보**다.

우선순위:

1. 생성과 runtime 실행이 성공해야 한다.
2. decode lane count mismatch가 없어야 한다.
3. steering mode mismatch가 없어야 한다.
4. lane 위치와 steer 차이가 작아야 한다.
5. 그중 latency가 가장 낮은 후보를 우선한다.

만약 quantized 후보가 전부 의미 보존에 실패하면, 12번으로 보내지 않고 양자화 설정을 다시 잡아야 한다.


In [11]:
valid = summary_df[summary_df["meaning_pass"] == True].copy()
if len(valid):
    selected = valid.sort_values("latency_pipeline_mean_ms").iloc[0].to_dict()
else:
    selected = None

calibration_sets = {
    f"{set_name}/{role_name}": int(count)
    for (set_name, role_name), count in calib_records.groupby(["set", "role"]).size().items()
}

report = {
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "notebook": "11_onnx_cpu_quantization_candidates_v1.ipynb",
    "source_onnx": str(SOURCE_ONNX),
    "source_package": str(PKG10),
    "calibration": {
        "records": int(len(calib_records)),
        "requires_gt": False,
        "sets": calibration_sets,
        "note": "Calibration uses only images to measure activation ranges; labels/GT are not used.",
    },
    "candidate_generation": candidate_generation.to_dict(orient="records"),
    "summary": summary_df.to_dict(orient="records"),
    "selected_for_pi_validation": selected,
    "next": [
        "Inspect candidate_eval_summary.csv.",
        "If selected_for_pi_validation is not null, create 12_quantized_pi_runtime_validation_v1.ipynb using that ONNX.",
        "Final acceptance must be based on Pi latency + parity, not local latency alone.",
    ],
}
write_json(REPORTS_DIR / "quantization_candidate_report.json", report)

if selected:
    write_json(REPORTS_DIR / "selected_candidate_for_pi.json", selected)
    print("Selected candidate for 12 Pi validation:")
    print(json.dumps(selected, indent=2, ensure_ascii=False))
else:
    print("No candidate passed the local meaning-preservation rules. Inspect the summary table.")

print("report:", REPORTS_DIR / "quantization_candidate_report.json")


No candidate passed the local meaning-preservation rules. Inspect the summary table.
report: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild\review_outputs\11_onnx_cpu_quantization_candidates_v1\reports\quantization_candidate_report.json


## 시각 검수 이미지

정량 지표가 통과해도 lane이 눈으로 이상하게 보이면 후보를 보내면 안 된다.  
아래 셀은 선택된 후보가 있을 때 field3 frame 일부에 대해 FP32 lane과 candidate lane을 함께 그린다.

- 초록: FP32 ONNX + 07 decoder
- 자홍: 후보 ONNX + 07 decoder

두 선이 거의 겹치면 양자화가 주행 의미를 잘 보존한 것이다.


In [12]:
def draw_lane_compare_sheet(selected_candidate, max_frames=12):
    if not selected_candidate:
        print("No selected candidate; skip visual check.")
        return None
    model_path = Path(selected_candidate["path"])
    cand_session, cand_input, cand_output = make_session_from_model(model_path)
    fp32_session, fp32_input, fp32_output = make_session_from_model(MODELS_DIR / SOURCE_ONNX.name)
    rows = parity_records[parity_records["set"] == "field3"].head(max_frames)
    tiles = []
    for _, rec in rows.iterrows():
        bgr = imread_bgr(rec["image_path"])
        vis = bgr.copy()
        fp32_raw = run_raw_from_session(fp32_session, fp32_input, fp32_output, bgr)
        cand_raw = run_raw_from_session(cand_session, cand_input, cand_output, bgr)
        fp32_lanes = decode_raw_to_lanes(fp32_raw, decode_contract)
        cand_lanes = decode_raw_to_lanes(cand_raw, decode_contract)
        for lane in fp32_lanes:
            pts = np.asarray(lane["points"], dtype=np.int32)
            if len(pts) >= 2:
                cv2.polylines(vis, [pts], False, (0, 220, 0), 3)
        for lane in cand_lanes:
            pts = np.asarray(lane["points"], dtype=np.int32)
            if len(pts) >= 2:
                cv2.polylines(vis, [pts], False, (255, 0, 255), 2)
        cv2.putText(vis, rec["key"], (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255,255,255), 2, cv2.LINE_AA)
        tile = cv2.resize(vis, (432, 324))
        tiles.append(tile)
    if not tiles:
        print("No tiles.")
        return None
    cols = 3
    rows_img = []
    for i in range(0, len(tiles), cols):
        chunk = tiles[i:i+cols]
        while len(chunk) < cols:
            chunk.append(np.zeros_like(tiles[0]))
        rows_img.append(np.concatenate(chunk, axis=1))
    sheet = np.concatenate(rows_img, axis=0)
    out = VIS_DIR / f"field3_fp32_vs_{selected_candidate['candidate']}.jpg"
    imwrite_bgr(out, sheet)
    print("visual sheet:", out)
    return out

visual_path = draw_lane_compare_sheet(selected)
visual_path


No selected candidate; skip visual check.


## Run 후 읽는 법

가장 먼저 볼 파일:

```text
review_outputs/11_onnx_cpu_quantization_candidates_v1/tables/candidate_eval_summary.csv
review_outputs/11_onnx_cpu_quantization_candidates_v1/reports/selected_candidate_for_pi.json
review_outputs/11_onnx_cpu_quantization_candidates_v1/reports/quantization_candidate_report.json
```

판단:

```text
meaning_pass=True 후보가 있고 latency가 FP32보다 줄었다
  → 12번 Pi quantized runtime 검증으로 진행

meaning_pass=True 후보가 있지만 latency 개선이 거의 없다
  → Pi에서 한 번은 재보되, 양자화 효과가 제한적일 가능성

모든 quantized 후보가 meaning_pass=False
  → quantization 설정을 바꾸거나, 입력 크기/모델 구조 최적화로 넘어가야 함
```

11번의 결론은 최종 배포가 아니다.  
**최종 판단은 12번에서 Pi ARM CPU로 같은 검증을 다시 돌린 뒤 내린다.**
